# Week 13 — Policies and Causal Inference

**Focus:** What can historical data identify about the consequence of an action?

*Live-coding notebook*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smc77/uc_finmlai/blob/main/lectures/week13/week13_demos.ipynb)

## What this notebook is about

Prediction asks what is likely to happen. Policy and causal questions ask what will happen because we choose an action. This notebook studies the additional assumptions needed to answer that harder question.

The first half checks logged-action support, compares exact dynamic programming with learned control, exposes a learn-to-trade search on pure noise, and solves execution by backward induction. The second half uses difference-in-differences, instrumental variables, and uplift modeling to estimate effects under different identification assumptions.

By the end, you should be able to:

- ask whether historical behavior supports the action a new policy wants to take;
- distinguish policy optimization from policy evaluation;
- explain why searching policies on one path can manufacture a winner;
- state the identifying assumption behind a causal estimate; and
- distinguish average effects from treatment targeting.

The simulations include known truth so that identification can be checked. A method working here does not remove the need to defend its assumptions in observational finance data.

Week 13 contains several deliberately different fixtures. The canonical execution demo matches Chapter 34 exactly. The collider and clustered-panel demos answer additional questions with separately named fixtures, so their numbers should not be compared with the preceding chapter figures as though they were reruns. The full parameter and purpose ledger is in `lectures/week13/lab/EXPERIMENT_REGISTRY.md`.

### Course context

For a concise review and the optional textbook route, see [Week 13 summary](https://github.com/smc77/uc_finmlai/blob/main/lectures/week13/week13_summary.md) and [Week 13 reading guide](https://github.com/smc77/uc_finmlai/blob/main/lectures/week13/reading.md). The notebook itself is designed to remain understandable without them.

## How to use this notebook

At the first break, run any **Setup** cell and then the **Imports** cell. If there is no Setup cell, begin with Imports. After that, use the slide cue to jump to the named demo; you do not need to rerun the whole notebook at every break. To check the whole notebook from a clean start, use **Runtime → Run all**.

Each demo follows the same rhythm: predict what the output should show, run the cell, and follow the task immediately underneath it. An **After you run** cell is editable: double-click it, replace `[write here]`, and press Shift-Enter. When an editable research-record entry appears, its completed comparison sits below it in a collapsed box—write your version before opening that box. This notebook is generated from the same source code the lecturer runs live.

## Jump to a demonstration

Both links jump within *this* notebook — neither opens a new tab. Use **Colab** when the notebook is open in Colab, **Jupyter** when it is open in Jupyter or rendered to HTML. You do not need to scroll through or rerun the whole notebook.

> If the **Jupyter** links do nothing in JupyterLab or Notebook 7, that is the windowed renderer, not a broken link: cells outside the viewport are not in the page, so there is no anchor to scroll to. Set *Settings → Settings Editor → Notebook → Windowing mode* to `defer` (or `none`) and they work.

### Deck A

<ul>
<li>Demo 0 — Logged-action support and effective sample size — <a href="#Demo-0-%E2%80%94-Logged-action-support-and-effective-sample-size">Jupyter</a> · <a href="#scrollTo=md-3fe7d4dfad3a">Colab</a></li>
<li>Demo 1 — Gridworld value iteration (RL on solid ground) — <a href="#Demo-1-%E2%80%94-Gridworld-value-iteration-(RL-on-solid-ground)">Jupyter</a> · <a href="#scrollTo=md-a1b07cbd4737">Colab</a></li>
<li>Demo 2 — &quot;Learn-to-trade&quot; fails: brute-force best-rule on pure noise — <a href="#Demo-2-%E2%80%94-%22Learn-to-trade%22-fails%3A-brute-force-best-rule-on-pure-noise">Jupyter</a> · <a href="#scrollTo=md-4be0020b3347">Colab</a></li>
<li>Demo 3 — Optimal execution: oracle, TWAP, and learned policy — <a href="#Demo-3-%E2%80%94-Optimal-execution%3A-oracle,-TWAP,-and-learned-policy">Jupyter</a> · <a href="#scrollTo=md-bc688e25e34b">Colab</a></li>
<li>Demo 3b — Stress frozen execution policies — <a href="#Demo-3b-%E2%80%94-Stress-frozen-execution-policies">Jupyter</a> · <a href="#scrollTo=md-e6d7a6d497ad">Colab</a></li>
</ul>

### Deck B

<ul>
<li>Demo 4 — Continuous-treatment collider challenge — <a href="#Demo-4-%E2%80%94-Continuous-treatment-collider-challenge">Jupyter</a> · <a href="#scrollTo=md-84b763d99a0f">Colab</a></li>
<li>Demo 5 — Seeded clustered-panel DiD challenge — <a href="#Demo-5-%E2%80%94-Seeded-clustered-panel-DiD-challenge">Jupyter</a> · <a href="#scrollTo=md-6ee2d169bff4">Colab</a></li>
<li>Demo 6 — IV: weak vs strong first stage — <a href="#Demo-6-%E2%80%94-IV%3A-weak-vs-strong-first-stage">Jupyter</a> · <a href="#scrollTo=md-c75831cccc78">Colab</a></li>
<li>Demo 7 — Uplift: T-learner on a simple sim — <a href="#Demo-7-%E2%80%94-Uplift%3A-T-learner-on-a-simple-sim">Jupyter</a> · <a href="#scrollTo=md-1f7052cfc669">Colab</a></li>
</ul>

## Read the code: choose an action, then check the comparison

A **policy** is a rule for choosing an action from the current situation. A
**causal effect** compares outcomes under different treatments. Both need evidence
about an alternative we did not observe, but they use different assumptions.

Run the setup once, then the cells in order. Demo 3b reuses Demo 3's frozen
policies. The other random fixtures reset their own generators, so changing the
number of logged rows cannot silently change the later causal examples. NumPy
makes the small worlds and their exact truth visible; these are teaching
constructions, not calibrated financial forecasts. The chapter figures and
notebook sometimes use different experiments; see the experiment registry.

## Project research-record checkpoint

Continue the one project-root `RESEARCH_RECORD.md`. This week adds two different forms of identification; use only the branch relevant to the project:

1. **RL branch:** behavior policy, action support, effective sample size, reward, simulator response, baseline, and independent evaluation.
2. **Causal branch:** unit, treatment, outcome horizon, estimand, assignment mechanism, interference, overlap, and target population.
3. **Uncertainty:** dependence-aware standard errors, intervals, sensitivity, or changed-simulator evidence.
4. **Boundary:** what is identified here, what requires an untestable assumption, and where the claim may not transport.
5. **Decision:** the action the estimate could justify and the fallback if identification or support fails.

Do not translate a retrospective association or regime label into a causal or predictive claim without recording the additional assumptions and forward evidence.

### Imports

> **Run this once before any demo.** It loads the packages and helper functions used below; there is no result to interpret in this cell.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io
from pathlib import Path
from sklearn.ensemble import GradientBoostingRegressor

rng = np.random.default_rng(42)

_demo_root = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
_figure_dir = _demo_root.parent / "figures" if _demo_root.name == "lab" and _demo_root.parent.name == "week13" else None

def save_teaching_figure(fig, name):
    """Keep local debrief figures identical to notebook output; no Colab writes."""
    if _figure_dir is None or not _figure_dir.exists():
        return
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=180, bbox_inches="tight")
    target = _figure_dir / name
    if not target.exists() or target.read_bytes() != buffer.getvalue():
        target.write_bytes(buffer.getvalue())

### Demo 0 — Logged-action support and effective sample size

> **Break cue:** Deck A, after recording segment S1. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

### 0. Did the log contain the choices we want to make?

**Start here:** four states and three actions form a 4-by-3 table. Each row of
`behavior` sums to one and gives the old rule's action probabilities. `candidate`
gives the new rule's probabilities. `rng.choice(3, p=behavior[s])` draws the
recorded action in state `s`.

`candidate[states, actions] / behavior[states, actions]` selects the probability
of each *recorded* action under each rule and divides them. In state 3, action 2
has probability .01 under the old rule and .60 under the new one, so its weight
is 60. `sum(weights)**2 / sum(weights**2)` measures weight concentration. It is
not a count of newly observed decisions or a causal-identification test.

The heatmap counts actual records; the red circle marks the rare pair. The bars
compare 10,000 rows with their weight-based effective sample size. Capping weights
at 10 or 20 can increase that diagnostic while changing the weighting and adding
bias. This cell has **no rewards**, so it does not estimate policy value.

**Predict, change, explain:** raise the candidate probability for the rare action
while keeping its row sum equal to one. Predict the effect on weight concentration.
Distinguish weak support from a truly zero logging probability; never divide by
zero and call the result an estimate. The chapter's zero-count schematic is a
separate example, not this positive-probability fixture.

In [ ]:
# Four states, three actions. The behavior policy scarcely takes action 2 in
# state 3, while the candidate policy asks for it often.
rng = np.random.default_rng(42)  # independent, reproducible support fixture
n_log = 10_000
states = rng.integers(0, 4, n_log)
behavior = np.array([
    [0.60, 0.35, 0.05],
    [0.55, 0.35, 0.10],
    [0.70, 0.28, 0.02],
    [0.74, 0.25, 0.01],
])
candidate = np.array([
    [0.25, 0.50, 0.25],
    [0.20, 0.45, 0.35],
    [0.20, 0.35, 0.45],
    [0.15, 0.25, 0.60],
])
actions = np.array([rng.choice(3, p=behavior[s]) for s in states])
weights = candidate[states, actions] / behavior[states, actions]
effective_n = weights.sum() ** 2 / np.square(weights).sum()
print("Logged-action support audit")
print(f"  nominal n:         {n_log:,}")
print(f"  maximum weight:    {weights.max():.1f}")
print(f"  effective n:       {effective_n:,.0f}")
print(f"  state 3/action 2:  behavior={behavior[3, 2]:.2%}, "
      f"candidate={candidate[3, 2]:.2%}")
# Expected: a nominal 10,000 rows collapse to an effective ~740 once the importance
#           weights are counted.


# A visible audit of the actual logged sample, not a schematic.
counts = np.zeros((4, 3), dtype=int)
np.add.at(counts, (states, actions), 1)
clipping_report = []
for cap in (10, 20):
    capped = np.minimum(weights, cap)
    clipped_fraction = np.mean(weights > cap)
    clipped_ess = capped.sum() ** 2 / np.square(capped).sum()
    clipping_report.append((cap, clipped_fraction, clipped_ess))
    print(f"  cap {cap}: {clipped_fraction:.2%} rows clipped; weight ESS={clipped_ess:.0f}")
print("Clipping changes the weights; increased weight ESS is not new evidence or an unbiased value estimate.")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
im = axes[0].imshow(counts, cmap="Blues")
for state in range(4):
    for action in range(3):
        axes[0].text(action, state, str(counts[state, action]), ha="center", va="center",
                     color="white" if counts[state, action] > counts.max()/2 else "black")
axes[0].set(xticks=range(3), yticks=range(4), xlabel="Logged action", ylabel="State",
            title="Actual counts in the 10,000-row log")
axes[0].scatter([2], [3], s=600, facecolors="none", edgecolors="#b33b32", linewidths=2)
axes[1].bar(["Recorded rows", "Weight ESS"], [n_log, effective_n], color=["#285a84", "#b33b32"])
for i, value in enumerate([n_log, effective_n]):
    axes[1].text(i, value+150, f"{value:,.0f}", ha="center")
axes[1].set(ylabel="Count / weight-concentration equivalent", ylim=(0, 11500),
            title="Rare choices concentrate the evidence")
fig.tight_layout(); save_teaching_figure(fig, "demo0_support.png"); plt.show()

### Demo 1 — Gridworld value iteration (RL on solid ground)

> **Break cue:** Deck A, after recording segment S1. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

### 1. Work backward from a reward — optional refresher

Five positions form a line. `step(s, a)` moves left or right; entering position 4
pays one and ends the episode. `V[s]` is the best discounted future reward from
position `s`. The update compares each action's immediate reward plus
`gamma * V[next_state]`, then keeps the larger value. `V_new` ensures every update
in a sweep uses the previous sweep's values.

The four nonterminal values become .729, .810, .900, and 1.000 when `gamma=.9`.
The goal's value is **zero**: the reward was paid on entry, so there is no second
reward after termination. This is value iteration using a supplied transition
model, not a policy learned from sampled experience. The execution example next
uses a finite deadline and works backward over time instead.

**Try:** set `gamma=.5`. Predict the four values before running. Do not add a
reward every time the agent remains at the terminal position.

In [ ]:
# A tiny MDP: 5 states in a line; enter state 4 to win (+1); otherwise 0.
# Show that value iteration converges and gives a sensible policy.
# This is the *kind* of problem RL is for — concrete reward, simulatable.

n_states = 5
gamma = 0.9
# transitions[s, a] -> s'; actions: 0 = left, 1 = right
def step(s, a):
    if a == 0:
        return max(0, s - 1)
    return min(n_states - 1, s + 1)

reward = np.zeros(n_states)
reward[-1] = 1.0  # only terminal state pays

V = np.zeros(n_states)
for sweep in range(200):
    V_new = V.copy()
    for s in range(n_states - 1):  # last state is terminal
        V_new[s] = max(
            reward[step(s, a)] + gamma * V[step(s, a)] for a in (0, 1)
        )
    if np.allclose(V, V_new):
        V = V_new
        print(f"converged after {sweep + 1} Bellman sweeps")
        break
    V = V_new

print("V(s) =", np.round(V, 3))
# Values before the goal are [.729, .81, .9, 1.0]; the terminal value is zero
# because the reward was paid on entry, not again after termination.
# Optimal policy: always go right. This is what RL is *good* at.

### Demo 2 — "Learn-to-trade" fails: brute-force best-rule on pure noise

> **Break cue:** Deck A, after recording segment S1. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

### 2. Search noise, then freeze the winner

`np.cumsum` turns independent Gaussian log returns into a price path. A rolling
mean uses only the available prefix. `signal[:-1] * np.diff(np.log(prices))`
uses yesterday's signal for the next return; omitting this lag leaks information.
`ma_pnl` sums signed log returns as a **gross trading score**, without transaction
costs, financing, or a cash-account wealth process.

The nested loops try short/long moving-average windows and retain the best
training score. The selected `best_a, best_b` are then held fixed on 30 new paths.
The histogram counts test paths; the red line is the selected training result,
and the green line is the independent-path mean. Test results must not choose a
new window pair. Thirty paths are an illustration, not a precise estimate of a
real strategy's expected return.

**Predict:** would trying more windows improve training performance even when
there is no signal? Change the declared search range, record its size, and compare
training versus independent paths. Keep the exercise exploratory; this is not an
untouched market test.

In [ ]:
# Generate price paths from random returns with NO signal.
# Pick the moving-average crossover (a, b) that maximizes profit in-sample.
# Deploy that exact rule on out-of-sample paths. Watch the gorgeous in-sample curve
# evaporate out of sample.

rng = np.random.default_rng(1302)
T = 500
prices_is = 100 * np.exp(np.cumsum(rng.standard_normal(T) * 0.01))

def ma_pnl(prices, a, b):
    if a >= b: return -np.inf
    sa = pd.Series(prices).rolling(a).mean()
    sb = pd.Series(prices).rolling(b).mean()
    signal = np.sign(sa - sb).fillna(0).values
    ret = np.diff(np.log(prices))
    return float((signal[:-1] * ret).sum())

best_a, best_b, best_pnl = 0, 0, -np.inf
for a in range(2, 30):
    for b in range(a + 1, 60):
        pnl = ma_pnl(prices_is, a, b)
        if pnl > best_pnl:
            best_pnl, best_a, best_b = pnl, a, b
print(f"In-sample best (a={best_a}, b={best_b}): cumulative log-PnL = {best_pnl:+.3f}")

# Deploy that exact rule on 30 out-of-sample paths
oos = []
for _ in range(30):
    p = 100 * np.exp(np.cumsum(rng.standard_normal(T) * 0.01))
    oos.append(ma_pnl(p, best_a, best_b))
oos = np.array(oos)
print(f"OOS mean PnL: {oos.mean():+.3f}   std: {oos.std():.3f}")
print(f"In-sample PnL was {best_pnl:+.3f}; the OOS distribution is centered near zero.")

fig, ax = plt.subplots(figsize=(8.2, 4.0))
bins = np.linspace(min(oos.min(), -0.18), max(oos.max(), best_pnl) + 0.025, 13)
ax.hist(oos, bins=bins, color="#4C78A8", alpha=0.82,
        edgecolor="white", label="30 independent paths")
ax.axvline(0.0, color="#333333", linewidth=1.4, linestyle=":",
           label="zero")
ax.axvline(best_pnl, color="#E45756", linewidth=2.4,
           label=f"selected training result = {best_pnl:+.3f}")
ax.axvline(oos.mean(), color="#2A9D8F", linewidth=2.0, linestyle="--",
           label=f"independent-path mean = {oos.mean():+.3f}")
ax.set(
    xlabel="Cumulative log P&L of the frozen moving-average rule",
    ylabel="Number of independent paths",
)
ax.legend(frameon=False, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
save_teaching_figure(fig, "demo2_policy_search.png")
plt.show()
# Expected: the in-sample best rule shows positive cumulative PnL; out of sample the
#           distribution is centered near zero.

### Mini-project: audit the log

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demos 0 and 2 — Logged-action support and the learn-to-trade search on pure noise**.

Run the policy-search and support cells in `week13_demos.ipynb`.

Produce:

1. one state-action pair with weak or absent support;
2. maximum weight and effective sample size;
3. the selected policy's training and out-of-sample result.

**Research record:** the state-action pair with weak support; maximum weight and effective sample size; the selected policy's training and out-of-sample results.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 3 — Optimal execution: oracle, TWAP, and learned policy

> **Break cue:** Deck A, after recording segment S2. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

### 3. Learn a selling rule and compare it with the exact answer

**Question:** how should we sell 12 units in six decisions? A state is
`(t, q, z)`: time, remaining inventory, and deep/thin liquidity. The action `x`
is an integer sale. `execution_cost` adds linear spread cost, quadratic temporary
impact, and a penalty for unsold inventory. Units are normalized simulator costs,
not dollars or basis points.

`feasible_actions(t, q)` allows sales from zero through `q`, but permits only `q`
at the deadline. That is a hard completion constraint. The evaluator rejects
policies that violate it. `execution_oracle` considers every legal sale and adds
its current cost to `P[z] @ value[t+1, q-x]`, an average over next liquidity states.
It works backward from the deadline. The value array is 7-by-13-by-2; the policy
array is 6-by-13-by-2.

Q-learning instead samples experience. Its 6-by-13-by-2-by-13 array stores
estimated remaining **cost** for each time, inventory, liquidity, and action.
Only feasible entries are compared. An epsilon-greedy rule sometimes tries a
random action; otherwise it picks `argmin`, because smaller cost is better.
The core update is:

```python
target = current_cost + best_estimated_next_cost
Q[state, action] += alpha * (target - Q[state, action])
```

At the final decision there is no next cost. `alpha` controls how far one
experience changes the estimate; epsilon controls exploration. Training starts
deep/thin equally often to broaden coverage. **Evaluation** uses the declared
60%/40% initial mixture and exact recursion, with learning switched off. The
chapter uses a different training recipe and 30 seeds; this notebook uses one
30,000-episode seed. The environment and cost contract match, not every training
setting.

TWAP spreads the remaining order across remaining decisions. Regret is candidate
expected cost minus the same world's exact oracle cost. Double `risk`, recompute
the oracle, and predict why its initial sale generally becomes larger. This
changes the objective, not the transition law. Neither low regret nor completion
inside this simulator establishes live-market performance.

In [ ]:
# This is the canonical Chapter 34 execution world: sell 12 normalized units
# in six decisions, with deep/thin liquidity, forced completion, and
# state-dependent spread and temporary impact.  Keep these parameters aligned
# with books/ch34/chapter.md and books/notebooks/build_ch34_notebook.py.
Q0 = 12
H = 6
initial_liquidity = np.array([0.6, 0.4])  # deep, thin
base_env = {
    "spread": np.array([0.04, 0.12]),
    "eta": np.array([0.06, 0.22]),
    "risk": 0.015,
    "P": np.array([[0.80, 0.20], [0.30, 0.70]]),
}


def feasible_actions(t, q):
    """The deadline mask permits only complete liquidation at the final step."""
    return np.array([q]) if t == H - 1 else np.arange(q + 1)


def execution_cost(q, x, z, env):
    return (env["spread"][z] * x
            + env["eta"][z] * x ** 2
            + env["risk"] * (q - x) ** 2)


def execution_oracle(env):
    """Exact finite-horizon backward induction for the declared environment."""
    value = np.zeros((H + 1, Q0 + 1, 2))
    policy = np.zeros((H, Q0 + 1, 2), dtype=int)
    for t in range(H - 1, -1, -1):
        for q in range(Q0 + 1):
            for z in range(2):
                actions = feasible_actions(t, q)
                action_values = []
                for x in actions:
                    continuation = 0.0
                    if t < H - 1:
                        continuation = env["P"][z] @ value[t + 1, q - x]
                    action_values.append(execution_cost(q, x, z, env) + continuation)
                best = int(np.argmin(action_values))
                value[t, q, z] = action_values[best]
                policy[t, q, z] = actions[best]
    return value, policy


def evaluate_execution_policy(policy, env):
    """Exact expected cost of a frozen policy under one environment law."""
    if policy.shape != (H, Q0 + 1, 2):
        raise ValueError("Policy shape must be (time, inventory, liquidity).")
    for t in range(H):
        for q in range(Q0 + 1):
            if any(x not in feasible_actions(t, q) for x in policy[t, q]):
                raise ValueError("Policy violates inventory or completion constraints.")
    value = np.zeros((H + 1, Q0 + 1, 2))
    for t in range(H - 1, -1, -1):
        for q in range(Q0 + 1):
            for z in range(2):
                x = int(policy[t, q, z])
                continuation = 0.0
                if t < H - 1:
                    continuation = env["P"][z] @ value[t + 1, q - x]
                value[t, q, z] = execution_cost(q, x, z, env) + continuation
    return float(initial_liquidity @ value[0, Q0])


def make_twap_policy():
    policy = np.zeros((H, Q0 + 1, 2), dtype=int)
    for t in range(H):
        periods_left = H - t
        for q in range(Q0 + 1):
            x = q if periods_left == 1 else int(np.ceil(q / periods_left))
            policy[t, q, :] = x
    return policy


def learn_execution_policy(env, episodes=30_000, seed=1313):
    """Tabular cost-minimizing Q-learning with epsilon-greedy exploration."""
    local_rng = np.random.default_rng(seed)
    q_values = np.zeros((H, Q0 + 1, 2, Q0 + 1))
    visits = np.zeros_like(q_values, dtype=int)
    for episode in range(episodes):
        q = Q0
        z = int(local_rng.integers(2))
        epsilon = max(0.03, 0.30 * (1 - episode / episodes))
        for t in range(H):
            actions = feasible_actions(t, q)
            if local_rng.random() < epsilon:
                x = int(local_rng.choice(actions))
            else:
                x = int(actions[np.argmin(q_values[t, q, z, actions])])
            target = execution_cost(q, x, z, env)
            q_next = q - x
            if t < H - 1:
                z_next = int(local_rng.choice(2, p=env["P"][z]))
                next_actions = feasible_actions(t + 1, q_next)
                target += np.min(q_values[t + 1, q_next, z_next, next_actions])
            else:
                z_next = z
            visits[t, q, z, x] += 1
            alpha = 0.35 / np.sqrt(1 + 0.002 * visits[t, q, z, x])
            q_values[t, q, z, x] += alpha * (target - q_values[t, q, z, x])
            q, z = q_next, z_next

    policy = np.zeros((H, Q0 + 1, 2), dtype=int)
    for t in range(H):
        for q in range(Q0 + 1):
            actions = feasible_actions(t, q)
            for z in range(2):
                policy[t, q, z] = int(actions[np.argmin(q_values[t, q, z, actions])])
    return policy, visits


_, oracle_policy = execution_oracle(base_env)
learned_policy, learned_visits = learn_execution_policy(base_env)
twap_policy = make_twap_policy()
oracle_cost = evaluate_execution_policy(oracle_policy, base_env)

print("Base execution world: expected cost and regret")
for label, policy in [
    ("exact oracle", oracle_policy),
    ("learned Q-policy", learned_policy),
    ("TWAP", twap_policy),
]:
    policy_cost = evaluate_execution_policy(policy, base_env)
    print(f"  {label:16s} cost={policy_cost:6.3f}  regret={policy_cost-oracle_cost:6.3f}")
print("  first action from 12 units [deep, thin]")
print(f"    exact:   {oracle_policy[0, Q0].tolist()}")
print(f"    learned: {learned_policy[0, Q0].tolist()}")

# One declared intervention in the objective: double inventory-risk cost.
high_risk_env = {**base_env, "risk": 2 * base_env["risk"]}
_, high_risk_oracle = execution_oracle(high_risk_env)
print("  after doubling inventory risk, exact first action [deep, thin]")
print(f"    before:  {oracle_policy[0, Q0].tolist()}")
print(f"    after:   {high_risk_oracle[0, Q0].tolist()}")

### Mini-project: meet the oracle

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demo 3 — Optimal execution: oracle, TWAP, and learned policy**.

Run the execution Markov decision process (MDP) cells in `week13_demos.ipynb`.

Compare the exact policy, time-weighted average price (TWAP), and one learned
policy. Report cost and regret, then change the inventory-risk penalty and
explain how the oracle's first action changes.

**Research record:** cost and regret for the exact policy, TWAP, and the learned
policy; the first deep- and thin-liquidity actions; the inventory-risk penalty
before and after; why the first action changes.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 3b — Stress frozen execution policies

> **Break cue:** Deck A, after recording segment S3. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

### 3b. Keep the policy fixed while changing the world

The arrays `oracle_policy`, `learned_policy`, and `twap_policy` are the base-world
rules from Demo 3. `stress_envs` changes liquidity persistence, temporary impact,
or a common per-unit spread charge. Each environment receives a **new oracle**;
the three candidate policies do not retrain.

The cell prints both absolute costs and regret. Adding .08 per unit charges all
complete 12-unit schedules exactly .96 more, so their regrets stay unchanged.
Higher impact or a changed transition matrix can change the preferred schedule.
Using the old oracle's cost as the new world's reference would mix these effects.

**Try:** calculate the .96 surcharge by hand and verify it for all three rules.
Then identify a stress that changes relative regret. These are parameter changes
inside one model. The lecture's matching-engine audit is a separate simulator
with incomplete fills and a different metric; it is not reproduced by this cell.

In [ ]:
# Recompute the oracle in each changed world, but keep the base-world candidate
# policies frozen. Regret therefore measures failure to adapt to the new law.
stress_envs = {
    "base": base_env,
    "persistent thin": {
        **base_env,
        "P": np.array([[0.65, 0.35], [0.10, 0.90]]),
    },
    "higher impact": {
        **base_env,
        "eta": 1.8 * base_env["eta"],
    },
    "common spread surcharge": {
        **base_env,
        "spread": base_env["spread"] + 0.08,
    },
}

print("Frozen-policy stress audit: regret above each changed-world oracle")
print("world                 base oracle   learned Q     TWAP")
stress_results = {}
for world, env in stress_envs.items():
    _, changed_oracle = execution_oracle(env)
    changed_oracle_cost = evaluate_execution_policy(changed_oracle, env)
    regrets = []
    costs = []
    for policy in (oracle_policy, learned_policy, twap_policy):
        cost = evaluate_execution_policy(policy, env)
        costs.append(cost)
        regrets.append(cost - changed_oracle_cost)
    print(f"{world:20s} {regrets[0]:11.3f} {regrets[1]:11.3f} {regrets[2]:8.3f}")
    print("  absolute costs:", np.round(costs, 3), "world oracle:", round(changed_oracle_cost, 3))
    stress_results[world] = {"costs": costs, "regrets": regrets, "oracle": changed_oracle_cost}

### Mini-project: environment audit

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demo 3b — Stress frozen execution policies**. Reuse the
weak-support state-action pair recorded in Demo 0; do not rerun Demo 0.

Write one page for the execution policy:

- observed evidence
- simulated response
- unsupported action
- stress result: absolute cost and regret in each changed world
- baseline
- authority warranted now

Keep it beside the final-project result you plan to bring next week.

**Research record:** observed evidence; simulated response; the unsupported
action; absolute cost and regret in each changed world; the baseline; the
authority the evidence warrants now.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 4 — Continuous-treatment collider challenge

> **Break cue:** Deck B, after recording segment S4. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

### 4. Choose controls from the causal story

`Z` and `U` both cause treatment and outcome. The planted treatment effect is .6.
`collider = .8*treatment + .8*outcome + noise` is a common consequence, measured
afterward. The simulator reveals `U`; a real researcher might not observe it.
This is a continuous-treatment example, unlike the chapter's binary-treatment
adjustment figure.

`np.column_stack([ones, treatment, *controls])` creates a design matrix with one
row per person. `lstsq(...)[0][1]` retrieves the treatment coefficient after the
intercept. The loop fits four specifications in 250 populations: no controls,
only Z, both common causes, then both causes plus the collider. Boxes show
variation across populations, not one dataset's confidence interval.

**Predict:** which specification should recover .6? Adding the collider can
associate treatment with outcome noise even after the common causes are held
fixed. More predictive inputs do not automatically create a better causal
comparison. Change the collider noise and explain the effect before rerunning.

In [ ]:
# This named collider challenge is deliberately different from the chapter's
# binary-treatment repeated adjustment audit. Two variables cause both a
# continuous treatment and its outcome. Correct adjustment recovers the
# planted effect; incomplete or post-treatment adjustment does not.
adjust_rng = np.random.default_rng(1304)
true_effect = 0.6
adjustment_results = {
    "unadjusted": [],
    "adjust Z only": [],
    "adjust Z and U": [],
    "add collider C": [],
}
for _ in range(250):
    n_adjust = 1_200
    Z = adjust_rng.standard_normal(n_adjust)
    U = adjust_rng.standard_normal(n_adjust)
    treatment = 0.8 * Z + 0.8 * U + adjust_rng.standard_normal(n_adjust)
    outcome = (true_effect * treatment + Z + U
               + adjust_rng.standard_normal(n_adjust))
    collider = (0.8 * treatment + 0.8 * outcome
                + adjust_rng.standard_normal(n_adjust))
    specifications = [[], [Z], [Z, U], [Z, U, collider]]
    for label, controls in zip(adjustment_results, specifications):
        design = np.column_stack([np.ones(n_adjust), treatment, *controls])
        coefficient = np.linalg.lstsq(design, outcome, rcond=None)[0][1]
        adjustment_results[label].append(coefficient)

print(f"Known-truth adjustment audit (true effect = {true_effect:.1f})")
for label, estimates in adjustment_results.items():
    print(f"  {label:18s} mean={np.mean(estimates):+6.3f}  sd={np.std(estimates):.3f}")
print("Only the sufficient pre-treatment adjustment set centers on the truth.")

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.boxplot(list(adjustment_results.values()), vert=False)
ax.set_yticks(np.arange(1, len(adjustment_results) + 1), list(adjustment_results))
ax.axvline(true_effect, color="darkorange", ls="--", label="Planted treatment effect")
ax.set(xlabel="Estimated treatment coefficient across 250 worlds",
       title="More controls are not always better controls")
ax.legend()
fig.tight_layout()
save_teaching_figure(fig, "demo4_collider_adjustment.png")
plt.show()

### Mini-project: causal graph

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demo 4 — Continuous-treatment collider challenge**, then
draw the causal graph before interpreting the estimate.

Record the planted truth and the four mean treatment coefficients.

Choose a financial intervention: a credit offer, a disclosure rule, or a
trading constraint.

Draw treatment, outcome, two plausible common causes, and one variable you
should not control for. State the estimand in one sentence.

**Research record:** the planted truth and four mean coefficients; the
intervention chosen; treatment, outcome, and the two plausible common causes;
the variable you must not control for and why; the estimand in one sentence.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Mini-project: draw before estimating

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demo 4 — Continuous-treatment collider challenge**, then
draw the causal graph before interpreting the estimate.

Choose a financial intervention: a credit offer, a disclosure rule, or a
trading constraint.

Draw treatment, outcome, two plausible common causes, and one variable you
should not control for. State the estimand in one sentence.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 5 — Seeded clustered-panel DiD challenge

> **Break cue:** Deck B, after recording segment S5. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

### 5. Separate a treatment effect from a different untreated trend

The panel has 200 firms and 60 months. Treatment is assigned by firm, starts at
month 30, and adds .3. Within-firm shocks persist: `.8*previous + new_noise`.
The outcome stacks 12,000 rows, but these are not 12,000 independent firms.

`treated * post` marks treated firms after adoption. The four-column regression
is `[1, treated, post, treated*post]`. This is the pooled two-group/two-period
contrast after averaging months. In this balanced common-date fixture its point
estimate equals the corresponding firm/time fixed-effects contrast; it is not
code for arbitrary staggered adoption.

The cluster calculation sums `X_f.T @ residual_f` **within each firm**, then adds
the outer products across firms. The resulting CR0 covariance has no finite-sample
multiplier. It allows within-firm dependence; it assumes firms are independent
for this calculation. It cannot repair a wrong untreated trend.

Adding `treated * t/(T-1) * .35` changes the untreated path without changing .3.
Compare the two mean-trajectory panels, then the event-time panels. The latter
subtract each firm's month-29 outcome, then compare average changes between
groups. Error bars use variation across firms and are pointwise approximate 95%
intervals, not simultaneous bands. The omitted month has zero estimate and
zero uncertainty by construction, not because that month provides infinite
information. Red lines show simulator truth; real data would not supply them.

**Predict, change, explain:** change .35 to zero or reverse its sign. Predict the
coefficient change. A longer pre-period can help reveal divergence; no pre-period
plot proves the missing post-treatment trend. The code implements common
adoption, not a treatment-timing or staggered-adoption experiment.

In [ ]:
# This named clustered-panel exercise is deliberately different from the
# chapter's repeated two-group DiD assumption audit. Simulate N firms over T
# months, with treatment at t = T/2 for half. Within-firm serial correlation
# inflates the apparent significance.

rng = np.random.default_rng(1305)
n_firms = 200
T = 60
treated = rng.random(n_firms) < 0.5
tau = 0.3  # true treatment effect

# Build the panel: firm fixed effect + AR(1) shock + treatment effect after t=30
firm_fe = rng.standard_normal(n_firms) * 0.5
rows = []
shocks = np.zeros(n_firms)
for t in range(T):
    shocks = 0.8 * shocks + rng.standard_normal(n_firms) * 0.3
    for i in range(n_firms):
        post = 1.0 if t >= T // 2 else 0.0
        effect = tau * post * (1 if treated[i] else 0)
        y = firm_fe[i] + shocks[i] + effect
        rows.append((i, t, int(treated[i]), int(post), y))

df = pd.DataFrame(rows, columns=["firm", "t", "treated", "post", "y"])

# Naive OLS DiD: y ~ treated + post + treated*post
df["did"] = df["treated"] * df["post"]
X = df[["treated", "post", "did"]].values
X = np.column_stack([np.ones(len(df)), X])
y = df["y"].values
beta, *_ = np.linalg.lstsq(X, y, rcond=None)
resid = y - X @ beta

# iid SEs
n, k = X.shape
sigma2 = resid @ resid / (n - k)
cov_iid = sigma2 * np.linalg.inv(X.T @ X)
se_iid = np.sqrt(np.diag(cov_iid))

# Cluster-robust at firm level (CR0 sandwich, no finite-sample multiplier)
XtX_inv = np.linalg.inv(X.T @ X)
S = np.zeros((k, k))
for f in df["firm"].unique():
    mask = (df["firm"] == f).values
    Xf = X[mask]
    rf = resid[mask]
    u = Xf.T @ rf
    S += np.outer(u, u)
cov_cr = XtX_inv @ S @ XtX_inv
se_cr = np.sqrt(np.diag(cov_cr))

print(f"True tau = {tau}; estimated = {beta[3]:.3f}")
print(f"iid SE:               {se_iid[3]:.4f}   t = {beta[3]/se_iid[3]:.2f}")
print(f"Cluster-robust SE:    {se_cr[3]:.4f}   t = {beta[3]/se_cr[3]:.2f}")
print(f"Cluster/iid SE ratio in this seeded fixture: {se_cr[3] / se_iid[3]:.2f}x")

# Plant a differential trend for treated firms. The DiD regression still runs;
# its comparison no longer represents the treated group's untreated path.
trend = df["treated"].to_numpy() * (df["t"].to_numpy() / (T - 1)) * 0.35
y_bad = y + trend
beta_bad, *_ = np.linalg.lstsq(X, y_bad, rcond=None)
print(f"With a treated-group differential trend: DiD = {beta_bad[3]:.3f}")
print(f"Deviation from planted effect: original={beta[3]-tau:+.3f}; trend violation={beta_bad[3]-tau:+.3f}")
print("A longer pre-period can reveal the divergence; it cannot prove the "
      "missing post-treatment paths would have stayed parallel.")
# Expected: the estimate lands near the true 0.3, while the cluster-robust
#           uncertainty is larger because repeated rows within a firm are dependent.

fig, axes = plt.subplots(1, 2, figsize=(10, 3.7), sharey=True)
for outcome, title, ax in [(y, "Planted parallel trends", axes[0]),
                           (y_bad, "Add an untreated trend difference", axes[1])]:
    plotted = df.assign(plotted_outcome=outcome)
    for status, label in [(0, "Comparison"), (1, "Treated")]:
        means = plotted[plotted.treated == status].groupby("t").plotted_outcome.mean()
        ax.plot(means.index, means, label=label)
    ax.axvline(T // 2, color="gray", ls="--", label="Treatment starts")
    ax.set(xlabel="Month", ylabel="Mean outcome", title=title)
    ax.legend(fontsize=8)
fig.tight_layout()
save_teaching_figure(fig, "demo5_did_trends.png")
plt.show()


# Event-time contrasts for this balanced, common-adoption panel.
# Each firm supplies its change relative to month 29; then compare the groups.
# This is the saturated common-date event study, not a staggered-adoption estimator.
def event_time_contrasts(outcomes):
    panel = df.assign(value=outcomes).pivot(index="firm", columns="t", values="value").to_numpy()
    changes = panel - panel[:, [T // 2 - 1]]
    estimate = changes[treated].mean(axis=0) - changes[~treated].mean(axis=0)
    se = np.sqrt(changes[treated].var(axis=0, ddof=1) / treated.sum()
                 + changes[~treated].var(axis=0, ddof=1) / (~treated).sum())
    return estimate, se

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for outcomes, title, ax in [(y, "Parallel untreated trends", axes[0]),
                            (y_bad, "Differential untreated trend", axes[1])]:
    event_est, event_se = event_time_contrasts(outcomes)
    event_time = np.arange(T) - T // 2
    ax.errorbar(event_time, event_est, yerr=1.96*event_se, fmt=".", capsize=2,
                color="#285a84", label="Estimate ± 1.96 SE")
    ax.axhline(0, color="gray", lw=1)
    ax.axvline(-.5, color="gray", ls="--")
    ax.plot(event_time, tau * (event_time >= 0), color="#b33b32", label="Planted treatment effect")
    ax.set(xlabel="Months relative to treatment (reference = −1)", ylabel="Group difference in changes", title=title)
    ax.legend(fontsize=8)
fig.tight_layout(); save_teaching_figure(fig, "demo5_event_study.png"); plt.show()
print("Event-study intervals are pointwise normal approximations across independent firms, not simultaneous bands.")

### Mini-project: break parallel trends

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demo 5 — Seeded clustered-panel DiD challenge**.

Run the DiD cell in `week13_demos.ipynb`.

1. estimate the baseline effect;
2. add a treated-group differential trend;
3. compare ordinary and cluster-robust uncertainty;
4. inspect the event study and explain which failure a longer pre-period could reveal.

**Research record:** the baseline effect; the effect after adding a treated-group
differential trend; ordinary against cluster-robust uncertainty; the event-study
warning that a longer pre-period could reveal.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 6 — IV: weak vs strong first stage

> **Break cue:** Deck B, after recording segment S6. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

### 6. Divide an outcome response by a treatment response

The instrument `Z` is independent of hidden cause `u` in this constructed world.
`X = pi*Z + .8*u + v` makes `pi` the instrument's strength. `Y = X + .8*u + noise`
plants effect 1.0, but OLS mixes the treatment effect with the shared cause.

The first regression estimates how much treatment changes with `Z`.
`X_hat = intercept_fs + pi_hat*Z` keeps only that instrument-related movement.
The second slope estimates outcome change per unit of that movement. With one
instrument and an intercept, it equals `cov(Z,Y)/cov(Z,X)`. A small denominator
makes the estimate unstable. The first-stage F is the squared slope t-statistic
using residual degrees of freedom `n-2`.

These are **two single draws**, not the chapter's 500-world distributions; their
planted effects also differ (1.0 here, .8 in the chapter). F measures relevance,
not exclusion or independence. **Try:** add `.5*Z` directly to Y. Predict why a
strong first stage then fails to identify the effect of X.

In [ ]:
# Y = beta * X + e ; X = pi * Z + confounder * u + v
rng = np.random.default_rng(1306)
n = 5000
iv_results = {}
for label, pi in [("STRONG (pi=1.0)", 1.0), ("WEAK   (pi=0.05)", 0.05)]:
    Z = rng.standard_normal(n)
    u = rng.standard_normal(n)  # unobserved confounder
    v = rng.standard_normal(n)
    X = pi * Z + 0.8 * u + v
    e = 0.8 * u + rng.standard_normal(n)
    Y = 1.0 * X + e  # true beta = 1.0; OLS will be biased upward by confounder

    # OLS (biased)
    beta_ols = np.cov(X, Y, ddof=0)[0, 1] / X.var()

    # 2SLS
    pi_hat = np.cov(Z, X, ddof=0)[0, 1] / Z.var()
    intercept_fs = X.mean() - pi_hat * Z.mean()
    X_hat = intercept_fs + pi_hat * Z
    # Exact homoskedastic first-stage F with an intercept: t^2 of pi_hat.
    resid_fs = X - X_hat
    se_pi = np.sqrt((resid_fs @ resid_fs / (n - 2)) / np.sum((Z - Z.mean()) ** 2))
    F = (pi_hat / se_pi) ** 2
    beta_iv = np.cov(X_hat, Y, ddof=0)[0, 1] / X_hat.var()

    iv_results[label] = {"ols": beta_ols, "iv": beta_iv, "F": F}
    print(f"{label}:  OLS beta = {beta_ols:.3f}   IV beta = {beta_iv:.3f}   F = {F:.0f}")
print("Truth = 1.0. Weak first stages make this ratio unstable; one draw cannot establish its bias.")
print("Relevance does not test exclusion or independence; those are planted assumptions here.")

### Demo 7 — Uplift: T-learner on a simple sim

> **Break cue:** Deck B, after recording segment S6. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

### 7. Rank who benefits, using a separate test population

`X` has three pre-treatment features per person. A fair coin assigns treatment.
Two `GradientBoostingRegressor` models fit the observed outcome separately for
treated and untreated people. This is a **T-learner**. Predictions
`mu1.predict(X_test) - mu0.predict(X_test)` rank estimated effects in 10,000 new
people; individual counterfactual outcomes remain unobserved.

The top ten percent is fixed before outcomes are inspected. With known treatment
probability .5, the observable contrast
`Y_test * (T_test/.5 - (1-T_test)/.5)` estimates effects on average. Its group mean
is noisy. Decile error bars show approximate pointwise uncertainty for the frozen
model's evaluation; they do not include variability from retraining the model.
The dashed truth line is available only because this is a simulation.

**Predict:** should a person with a high baseline outcome always rank highly for
treatment? Compare the baseline and effect formulas. Change the model using
training/validation data, then use a fresh test population for a new final claim.
Do not tune the treatment fraction to the peak of the final test curve. The
chapter's quadratic T-learner and this boosted-tree T-learner are different
fixtures, both evaluated using independent randomized outcomes.

In [ ]:
rng = np.random.default_rng(1307)
n = 4000
X = rng.standard_normal((n, 3))
T = (rng.random(n) < 0.5).astype(int)
# Baseline outcome plus heterogeneous treatment effect
tau_true = 0.5 * (X[:, 0] > 0).astype(float) - 0.3 * (X[:, 1] > 0.5).astype(float)
Y = 0.5 * X[:, 0] + 0.2 * X[:, 2] + T * tau_true + rng.standard_normal(n) * 0.5

mask_t = T == 1
mask_c = T == 0
mu1 = GradientBoostingRegressor(max_depth=3, random_state=13).fit(X[mask_t], Y[mask_t])
mu0 = GradientBoostingRegressor(max_depth=3, random_state=13).fit(X[mask_c], Y[mask_c])

# Freeze both fitted models and rank independent evaluation units.
n_test = 10_000
X_test = rng.standard_normal((n_test, 3))
T_test = (rng.random(n_test) < 0.5).astype(int)
tau_test = 0.5 * (X_test[:, 0] > 0) - 0.3 * (X_test[:, 1] > 0.5)
Y_test = (0.5 * X_test[:, 0] + 0.2 * X_test[:, 2]
          + T_test * tau_test + rng.standard_normal(n_test) * 0.5)
tau_hat = mu1.predict(X_test) - mu0.predict(X_test)

# Randomization probability is known. This observable contrast estimates
# the average effect in a frozen selected group; individual effects remain hidden.
ipw_effect = Y_test * (T_test / 0.5 - (1 - T_test) / 0.5)
order = np.argsort(-tau_hat)
top = order[:n_test // 10]  # fraction declared before viewing outcomes
print(f"Top-10% held-out effect estimate (IPW): {ipw_effect[top].mean():+.3f}")
print(f"Population held-out effect estimate:   {ipw_effect.mean():+.3f}")
print(f"Simulator-only check, top-10% truth:   {tau_test[top].mean():+.3f}")
print("These estimates have sampling uncertainty; do not retune the selected fraction on test outcomes.")

deciles = np.array_split(order, 10)
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.errorbar(range(1, 11), [ipw_effect[idx].mean() for idx in deciles],
            yerr=[1.96 * ipw_effect[idx].std(ddof=1) / np.sqrt(len(idx)) for idx in deciles],
            fmt="o-", capsize=3, label="Held-out IPW estimate ± 1.96 SE")
ax.plot(range(1, 11), [tau_test[idx].mean() for idx in deciles], "s--", label="Simulator truth")
ax.axhline(0, color="gray", lw=1)
ax.set(xlabel="Predicted-effect group (1 = highest)", ylabel="Average treatment effect",
       title="Does the frozen ranking find larger effects in new units?")
ax.legend()
fig.tight_layout()
save_teaching_figure(fig, "demo7_uplift.png")
plt.show()

## Where this leaves us

Actions require support, counterfactual reasoning, and explicit identifying assumptions. Neither a high policy value nor a causal coefficient interprets itself; the claim is only as strong as the behavior data, design, and decision boundary behind it.

Week 14 asks who is affected by financial models, how disparities and explanations are measured, and how performance is monitored after deployment.